In [4]:
from __future__ import annotations

import json
from itertools import combinations
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, f1_score


def nurse_id_from_path(path: Path) -> str:
    return path.stem.replace("processed_nurse_", "")


def infer_window_steps(time_values: np.ndarray, window_seconds: float) -> int:
    if len(time_values) < 2:
        return 1
    diffs = np.diff(time_values)
    positive_diffs = diffs[diffs > 0]
    if len(positive_diffs) == 0:
        return 1
    dt = float(np.median(positive_diffs))
    return max(1, int(round(window_seconds / dt)))


def build_windows(
    X: np.ndarray,
    y: np.ndarray,
    window_steps: int,
    stride_steps: int,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    if len(X) < window_steps:
        return np.empty((0, X.shape[1] * 6)), np.empty((0,), dtype=int), np.empty((0,), dtype=int)

    X_windows: list[np.ndarray] = []
    y_windows: list[int] = []
    end_indices: list[int] = []

    for start in range(0, len(X) - window_steps + 1, stride_steps):
        end = start + window_steps
        w = X[start:end]
        # Summarize local behavior over the window to keep dimensions compact.
        mean = w.mean(axis=0)
        std = w.std(axis=0)
        minv = w.min(axis=0)
        maxv = w.max(axis=0)
        last = w[-1]
        slope = (w[-1] - w[0]) / max(1, window_steps - 1)
        X_windows.append(np.concatenate([mean, std, minv, maxv, last, slope], axis=0))
        # Predict stress at the current endpoint from recent history.
        y_windows.append(int(y[end - 1]))
        end_indices.append(end - 1)

    return np.asarray(X_windows), np.asarray(y_windows), np.asarray(end_indices)


def load_and_process_nurse(
    csv_path: Path,
    window_seconds: float,
    fixed_window_steps: int | None,
    stride_fraction: float,
    test_ratio: float,
) -> dict[str, object] | None:
    df = pd.read_csv(csv_path)

    required_cols = {"time", "acc_mag", "EDA", "HR", "TEMP", "label"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"{csv_path.name} is missing columns: {sorted(missing)}")

    df = df.sort_values("time").reset_index(drop=True)
    df["label_bin"] = (df["label"] > 0).astype(int)

    features = ["acc_mag", "EDA", "HR", "TEMP"]
    X = df[features].to_numpy(dtype=np.float32)
    y = df["label_bin"].to_numpy(dtype=np.int8)

    window_steps = (
        fixed_window_steps
        if fixed_window_steps is not None
        else infer_window_steps(df["time"].to_numpy(dtype=np.float32), window_seconds)
    )
    stride_steps = max(1, int(round(window_steps * stride_fraction)))

    Xw, yw, end_idx = build_windows(X, y, window_steps, stride_steps)
    if len(Xw) == 0:
        return None

    split_idx = int(len(Xw) * (1.0 - test_ratio))
    split_idx = min(max(split_idx, 1), len(Xw) - 1)

    nurse_id = nurse_id_from_path(csv_path)
    return {
        "nurse_id": nurse_id,
        "window_steps": int(window_steps),
        "stride_steps": int(stride_steps),
        "num_windows": int(len(Xw)),
        "X_train": Xw[:split_idx],
        "y_train": yw[:split_idx],
        "X_test": Xw[split_idx:],
        "y_test": yw[split_idx:],
        "test_end_idx": end_idx[split_idx:],
        "test_times": df.loc[end_idx[split_idx:], "time"].to_numpy(dtype=np.float32),
    }

In [7]:
# Configuration
data_dir = Path("data/Eric/")  # Update this to your CSV directory
window_seconds = 30.0
fixed_window_steps = None  # Set to int if you want fixed windows, otherwise auto-infer
stride_fraction = 0.5
test_ratio = 0.2

# Load and preprocess all patient CSVs
csv_files = sorted(data_dir.glob("processed_nurse_*.csv"))
print(f"Found {len(csv_files)} CSV files")

all_nurse_data = {}
for csv_path in csv_files:
    try:
        result = load_and_process_nurse(
            csv_path,
            window_seconds=window_seconds,
            fixed_window_steps=fixed_window_steps,
            stride_fraction=stride_fraction,
            test_ratio=test_ratio,
        )
        if result is not None:
            nurse_id = result["nurse_id"]
            all_nurse_data[nurse_id] = result
            print(f"✓ Loaded nurse {nurse_id}: {result['num_windows']} windows")
        else:
            print(f"✗ Skipped {csv_path.name}: insufficient data")
    except Exception as e:
        print(f"✗ Error loading {csv_path.name}: {e}")

print(f"\nTotal nurses loaded: {len(all_nurse_data)}")

# Create folds with 2 nurses per test set
nurse_ids = list(all_nurse_data.keys())
folds = []

for fold_idx, (test_nurse_1, test_nurse_2) in enumerate(combinations(nurse_ids, 2)):
    train_nurses = [n for n in nurse_ids if n not in (test_nurse_1, test_nurse_2)]
    
    # Combine training data
    X_train = np.vstack([all_nurse_data[n]["X_train"] for n in train_nurses])
    y_train = np.concatenate([all_nurse_data[n]["y_train"] for n in train_nurses])
    
    # Combine test data
    X_test = np.vstack([all_nurse_data[test_nurse_1]["X_test"], all_nurse_data[test_nurse_2]["X_test"]])
    y_test = np.concatenate([all_nurse_data[test_nurse_1]["y_test"], all_nurse_data[test_nurse_2]["y_test"]])
    
    folds.append({
        "fold_idx": fold_idx,
        "test_nurses": (test_nurse_1, test_nurse_2),
        "train_nurses": train_nurses,
        "X_train": X_train,
        "y_train": y_train,
        "X_test": X_test,
        "y_test": y_test,
    })

print(f"\nCreated {len(folds)} folds")
print(f"Example fold 0: test nurses {folds[0]['test_nurses']}, train nurses: {len(folds[0]['train_nurses'])}")

Found 15 CSV files
✓ Loaded nurse 15: 617 windows
✓ Loaded nurse 5C: 1730 windows
✓ Loaded nurse 6B: 1650 windows
✓ Loaded nurse 6D: 1181 windows
✓ Loaded nurse 7A: 2753 windows
✓ Loaded nurse 7E: 505 windows
✓ Loaded nurse 83: 2744 windows
✓ Loaded nurse 8B: 846 windows
✓ Loaded nurse 94: 1171 windows
✓ Loaded nurse BG: 1216 windows
✓ Loaded nurse CE: 1627 windows
✓ Loaded nurse DF: 1815 windows
✓ Loaded nurse E4: 2974 windows
✓ Loaded nurse EG: 1097 windows
✓ Loaded nurse F5: 1070 windows

Total nurses loaded: 15

Created 105 folds
Example fold 0: test nurses ('15', '5C'), train nurses: 13


In [18]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

def evaluate(model, data_loader, criterion, device):
    """Evaluate model on a dataset."""
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            logits = model(X_batch).squeeze()
            loss = criterion(logits, y_batch.float())
            total_loss += loss.item() * len(X_batch)
            
            preds = (torch.sigmoid(logits) > 0.5).detach().cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(y_batch.detach().cpu().numpy())
    
    avg_loss = total_loss / len(data_loader.dataset)
    accuracy = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, zero_division=0)
    
    return avg_loss, accuracy, f1, all_preds, all_labels

class StressPredictor(nn.Module):
    """Binary classification MLP for stress prediction."""
    def __init__(self, input_size=24, hidden_sizes=None):
        super().__init__()
        if hidden_sizes is None:
            hidden_sizes = [64, 32]
        
        layers = []
        prev_size = input_size
        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(0.3))
            prev_size = hidden_size
        
        layers.append(nn.Linear(prev_size, 1))
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.network(x)


def train_epoch(model, train_loader, optimizer, criterion, device):
    """Train for one epoch."""
    model.train()
    total_loss = 0.0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        optimizer.zero_grad()
        logits = model(X_batch).squeeze()
        loss = criterion(logits, y_batch.float())
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * len(X_batch)
    
    return total_loss / len(train_loader.dataset)


def train_fold(fold, epochs=100, batch_size=32, learning_rate=0.001, device='cpu'):
    """Train and evaluate model on a single fold."""
    # Prepare data loaders
    X_train = torch.FloatTensor(fold["X_train"])
    y_train = torch.LongTensor(fold["y_train"])
    X_test = torch.FloatTensor(fold["X_test"])
    y_test = torch.LongTensor(fold["y_test"])
    
    train_dataset = TensorDataset(X_train, y_train)
    test_dataset = TensorDataset(X_test, y_test)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    # Initialize model
    model = StressPredictor(input_size=24, hidden_sizes=[64, 32]).to(device)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.BCEWithLogitsLoss()
    
    # Training loop
    best_test_f1 = 0.0
    patience_counter = 0
    patience = 15
    
    for epoch in range(epochs):
        train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
        test_loss, test_acc, test_f1, _, _ = evaluate(model, test_loader, criterion, device)
        
        if (epoch + 1) % 20 == 0:
            print(f"  Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Test Loss: {test_loss:.4f} | Test F1: {test_f1:.4f}")
        
        # Early stopping based on F1
        if test_f1 > best_test_f1:
            best_test_f1 = test_f1
            patience_counter = 0
        else:
            patience_counter += 1
        
        if patience_counter >= patience:
            print(f"  Early stopping at epoch {epoch+1}")
            break
    
    # Final evaluation
    test_loss, test_acc, test_f1, test_preds, test_labels = evaluate(model, test_loader, criterion, device)
    
    return {
        "fold_idx": fold["fold_idx"],
        "test_nurses": fold["test_nurses"],
        "test_loss": test_loss,
        "test_accuracy": test_acc,
        "test_f1": test_f1,
        "test_preds": test_preds,
        "test_labels": test_labels,
        "model": model,
    }


# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}\n")

# Train model on each fold
results = []
for fold in folds:
    print(f"Training fold {fold['fold_idx']} (test nurses: {fold['test_nurses']})")
    result = train_fold(fold, epochs=100, batch_size=32, learning_rate=0.001, device=device)
    results.append(result)
    print(f"  → Test Accuracy: {result['test_accuracy']:.4f}, F1: {result['test_f1']:.4f}\n")

# Summary
print("\n" + "="*60)
print("CROSS-VALIDATION RESULTS")
print("="*60)
summary_df = pd.DataFrame([
    {
        "Fold": r["fold_idx"],
        "Test Nurses": str(r["test_nurses"]),
        "Accuracy": f"{r['test_accuracy']:.4f}",
        "F1 Score": f"{r['test_f1']:.4f}",
        "Loss": f"{r['test_loss']:.4f}",
    }
    for r in results
])
print(summary_df.to_string(index=False))
print(f"\nMean F1: {np.mean([r['test_f1'] for r in results]):.4f}")
print(f"Mean Accuracy: {np.mean([r['test_accuracy'] for r in results]):.4f}")

Using device: cpu

Training fold 0 (test nurses: ('15', '5C'))


RuntimeError: Numpy is not available

In [22]:
def evaluate(model, data_loader, criterion, device):
    """Evaluate model on a dataset."""
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            logits = model(X_batch).squeeze()
            loss = criterion(logits, y_batch.float())
            total_loss += loss.item() * len(X_batch)
            
            preds = (torch.sigmoid(logits) > 0.5).detach().cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(y_batch.detach().cpu().numpy())
    
    avg_loss = total_loss / len(data_loader.dataset)
    accuracy = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, zero_division=0)
    
    return avg_loss, accuracy, f1, all_preds, all_labels

In [24]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader


class StressPredictor(nn.Module):
    """Binary classification MLP for stress prediction."""
    def __init__(self, input_size=24, hidden_sizes=None):
        super().__init__()
        if hidden_sizes is None:
            hidden_sizes = [64, 32]
        
        layers = []
        prev_size = input_size
        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(0.3))
            prev_size = hidden_size
        
        layers.append(nn.Linear(prev_size, 1))
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.network(x)


def train_epoch(model, train_loader, optimizer, criterion, device):
    """Train for one epoch."""
    model.train()
    total_loss = 0.0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        optimizer.zero_grad()
        logits = model(X_batch).squeeze()
        loss = criterion(logits, y_batch.float())
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * len(X_batch)
    
    return total_loss / len(train_loader.dataset)


def evaluate(model, data_loader, criterion, device):
    """Evaluate model on a dataset."""
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            logits = model(X_batch).squeeze()
            loss = criterion(logits, y_batch.float())
            total_loss += loss.item() * len(X_batch)
            preds = (torch.sigmoid(logits) > 0.5).int().detach().cpu().tolist()

            all_preds.extend(preds)
            all_labels.extend(y_batch.detach().cpu().tolist())
    
    avg_loss = total_loss / len(data_loader.dataset)
    accuracy = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, zero_division=0)
    
    return avg_loss, accuracy, f1, all_preds, all_labels



def train_fold(fold, epochs=100, batch_size=32, learning_rate=0.001, device='cpu'):
    """Train and evaluate model on a single fold."""
    # Prepare data loaders
    X_train = torch.FloatTensor(fold["X_train"])
    y_train = torch.LongTensor(fold["y_train"])
    X_test = torch.FloatTensor(fold["X_test"])
    y_test = torch.LongTensor(fold["y_test"])
    
    train_dataset = TensorDataset(X_train, y_train)
    test_dataset = TensorDataset(X_test, y_test)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    # Initialize model
    model = StressPredictor(input_size=24, hidden_sizes=[64, 32]).to(device)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.BCEWithLogitsLoss()
    
    # Training loop
    best_test_f1 = 0.0
    patience_counter = 0
    patience = 15
    
    for epoch in range(epochs):
        train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
        test_loss, test_acc, test_f1, _, _ = evaluate(model, test_loader, criterion, device)
        
        if (epoch + 1) % 20 == 0:
            print(f"  Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Test Loss: {test_loss:.4f} | Test F1: {test_f1:.4f}")
        
        # Early stopping based on F1
        if test_f1 > best_test_f1:
            best_test_f1 = test_f1
            patience_counter = 0
        else:
            patience_counter += 1
        
        if patience_counter >= patience:
            print(f"  Early stopping at epoch {epoch+1}")
            break
    
    # Final evaluation
    test_loss, test_acc, test_f1, test_preds, test_labels = evaluate(model, test_loader, criterion, device)
    
    return {
        "fold_idx": fold["fold_idx"],
        "test_nurses": fold["test_nurses"],
        "test_loss": test_loss,
        "test_accuracy": test_acc,
        "test_f1": test_f1,
        "test_preds": test_preds,
        "test_labels": test_labels,
        "model": model,
    }


# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}\n")

# Train model on each fold
results = []
for fold in folds:
    print(f"Training fold {fold['fold_idx']} (test nurses: {fold['test_nurses']})")
    result = train_fold(fold, epochs=100, batch_size=32, learning_rate=0.001, device=device)
    results.append(result)
    print(f"  → Test Accuracy: {result['test_accuracy']:.4f}, F1: {result['test_f1']:.4f}\n")

# Summary
print("\n" + "="*60)
print("CROSS-VALIDATION RESULTS")
print("="*60)
summary_df = pd.DataFrame([
    {
        "Fold": r["fold_idx"],
        "Test Nurses": str(r["test_nurses"]),
        "Accuracy": f"{r['test_accuracy']:.4f}",
        "F1 Score": f"{r['test_f1']:.4f}",
        "Loss": f"{r['test_loss']:.4f}",
    }
    for r in results
])
print(summary_df.to_string(index=False))
mean_f1 = sum(r['test_f1'] for r in results) / len(results)
mean_acc = sum(r['test_accuracy'] for r in results) / len(results)
print(f"\nMean F1: {mean_f1:.4f}")
print(f"Mean Accuracy: {mean_acc:.4f}")

Using device: cpu

Training fold 0 (test nurses: ('15', '5C'))
  Early stopping at epoch 16
  → Test Accuracy: 0.9638, F1: 0.9816

Training fold 1 (test nurses: ('15', '6B'))
  Early stopping at epoch 16
  → Test Accuracy: 1.0000, F1: 1.0000

Training fold 2 (test nurses: ('15', '6D'))
  Early stopping at epoch 16
  → Test Accuracy: 0.3435, F1: 0.5113

Training fold 3 (test nurses: ('15', '7A'))
  Early stopping at epoch 16
  → Test Accuracy: 0.9437, F1: 0.9710

Training fold 4 (test nurses: ('15', '7E'))


ValueError: Target size (torch.Size([1])) must be the same as input size (torch.Size([]))